You can also follow along on Google Colab!

<a target="_blank" href="https://colab.research.google.com/github/MadryLab/context-cite/blob/main/notebooks/quickstart_example.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Quickstart example for `ContextCite`

In this notebook, we'll provide an overview of the ContextCite API by going through a simple example. **If running in Colab, be sure to change your to a GPU runtime!**

Let's start by installing the library:

In [ ]:
# !pip install context-cite
# !pip install accelerate
# !pip install transformers==4.38.2 --upgrade --force-reinstall
!pip install --upgrade transformers

# !pip uninstall -y numpy transformers tokenizers torch

  Using cached transformers-4.51.3-py3-none-any.whl.metadata (38 kB)
  Using cached tokenizers-0.21.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
Using cached transformers-4.51.3-py3-none-any.whl (10.4 MB)
Using cached tokenizers-0.21.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.0 MB)
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2
  You can safely remove it manually.
  Attempting uninstall: transformers
    Found existing installation: transformers 4.38.2
    Uninstalling transformers-4.38.2:
      Successfully uninstalled transformers-4.38.2


In [1]:
import accelerate
print(accelerate.__version__)

import transformers
print(transformers.__version__)


/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1.6.0
4.51.3


We will use the `ContextCiter` class to attribute models' responses to sources within the context we provide to them.

In [ ]:
import context_cite
from transformers import AutoTokenizer

from context_cite import ContextCiter

In [3]:
# Load model directly
from transformers import AutoProcessor, AutoModel

processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-Omni-7B")
model = AutoModel.from_pretrained("Qwen/Qwen2.5-Omni-7B")

ValueError: The checkpoint you are trying to load has model type `qwen2_5_omni` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

You can update Transformers with the command `pip install --upgrade transformers`. If this does not work, and the checkpoint is very new, then there may not be a release version that supports this model yet. In this case, you can get the most up-to-date code by installing Transformers from source with the command `pip install git+https://github.com/huggingface/transformers.git`

For this example, we'll use a TinyLlama chat model.

In [2]:
import pandas as pd

# Load the CSV file
df_op4 = pd.read_csv("medbullets_op4.csv")

# Count duplicates before removing
num_duplicates = df_op4.duplicated(subset=['question']).sum()

# Remove duplicate rows based on the 'question' column, keeping the first occurrence
df_op4 = df_op4.drop_duplicates(subset=['question'], keep='first')

# Display the number of duplicate rows found
print(f"Number of duplicate rows removed: {num_duplicates}")

# Display the first few rows after removing duplicates
# print(df_op4.columns)
# print(df_op4.head())
print(len(df_op4))

Number of duplicate rows removed: 10
298


In [16]:
model_name_or_path = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B" ##TinyLlama/TinyLlama-1.1B-Chat-v1.0

context = """
Attention Is All You Need

Abstract
The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder. The best performing models also connect the encoder and decoder through an attention mechanism. We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. Experiments on two machine translation tasks show these models to be superior in quality while being more parallelizable and requiring significantly less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature. We show that the Transformer generalizes well to other tasks by applying it successfully to English constituency parsing both with large and limited training data.
1 Introduction
Recurrent neural networks, long short-term memory [13] and gated recurrent [7] neural networks in particular, have been firmly established as state of the art approaches in sequence modeling and transduction problems such as language modeling and machine translation [35, 2, 5]. Numerous efforts have since continued to push the boundaries of recurrent language models and encoder-decoder architectures [38, 24, 15].
Recurrent models typically factor computation along the symbol positions of the input and output sequences. Aligning the positions to steps in computation time, they generate a sequence of hidden states ht, as a function of the previous hidden state ht-1 and the input for position t. This inherently sequential nature precludes parallelization within training examples, which becomes critical at longer sequence lengths, as memory constraints limit batching across examples. Recent work has achieved significant improvements in computational efficiency through factorization tricks [21] and conditional computation [32], while also improving model performance in case of the latter. The fundamental constraint of sequential computation, however, remains.
Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 19]. In all but a few cases [27], however, such attention mechanisms are used in conjunction with a recurrent network.
In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output. The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.
"""
query = "What type of GPUs did the authors use in this paper?"

In [3]:
import re

def split_question(question_text):
    """Split question text into context and query (last sentence)."""
    sentences = re.split(r'(?<=[.?!])\s+', question_text.strip())
    if len(sentences) <= 1:
        return "", question_text.strip()
    context = " ".join(sentences[:-1])
    query = sentences[-1]
    return context, query

# Loop through each row and construct the input
for idx, row in df_op4.iterrows():
    full_question = row['question']
    opa, opb, opc, opd = row['opa'], row['opb'], row['opc'], row['opd']

    # Split into context and query
    context_text, query_text = split_question(full_question)

    # Combine query with answer options
    query_full = f"{query_text}\nA. {opa}\nB. {opb}\nC. {opc}\nD. {opd}"

    # Optional: show result for one row
    print(f"\n--- Row {idx} ---")
    print("Context:\n", context_text)
    print("Query:\n", query_full)

    # You could now pass `context_text` and `query_full` to ContextCiter, e.g.:
    # response = context_citer.generate_response(context=context_text, query=query_full)

    # Break early to preview just the first result
    # Stop after 5 rows
    if idx >= 4:
        break



--- Row 0 ---
Context:
 A 42-year-old woman is enrolled in a randomized controlled trial to study cardiac function in the setting of several different drugs. She is started on verapamil and instructed to exercise at 50% of her VO2 max while several cardiac parameters are being measured.
Query:
 During this experiment, which of the following represents the relative conduction speed through the heart from fastest to slowest?
A. AV node > ventricles > atria > Purkinje fibers
B. Purkinje fibers > ventricles > atria > AV node
C. Purkinje fibers > atria > ventricles > AV node
D. Purkinje fibers > AV node > ventricles > atria

--- Row 1 ---
Context:
 A 9-year-old girl presents to the emergency department with a fever and a change in her behavior. She presented with similar symptoms 6 weeks ago and was treated for an Escherchia coli infection. She also was treated for a urinary tract infection 10 weeks ago. Her mother says that last night her daughter felt ill, and her condition has been wors

In [4]:
model_id = "deepseek-ai/DeepSeek-R1-Distill-Qwen-14B"

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM    
model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto"  # ✅ valid here
    )
tokenizer = AutoTokenizer.from_pretrained(model_id)

/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|████████████████████████████████████████████████████████████████| 4/4 [01:42<00:00, 25.57s/it]
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/huggingface_hub/file_download.py:896: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [10]:
import os
import re
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM    

# Create output folder if it doesn't exist
output_dir = "Attribution Scores_14B"
os.makedirs(output_dir, exist_ok=True)

# Model to use
# model_name_or_path = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"

# Loop over the first 10 rows of df_op4
for idx, row in df_op4.head(298).iterrows():
    full_question = row["question"]
    opa, opb, opc, opd = row["opa"], row["opb"], row["opc"], row["opd"]

    # Split context and query
    context_text, query_text = split_question(full_question)

    # Format the full query prompt
    query_full = (
        f"{query_text}\n"
        f"A. {opa}\nB. {opb}\nC. {opc}\nD. {opd}\n\n"
        "Read the question and state your answer. "
        "State your answer, starting with 'Answer:', ending with two line breaks.\n\n"
    )

    try:
#         # Generate attribution results and response
#         cc = ContextCiter.from_pretrained(
#             model_name_or_path,
#             context=context_text,
#             query=query_full,
#             generate_kwargs={"max_new_tokens": 2048, "do_sample": False} #, "device_map": "auto"
#         )
        # Generate attribution results and response
        cc = ContextCiter(
            model,
            tokenizer,
            context=context_text,
            query=query_full,
            generate_kwargs={"max_new_tokens": 2048, "do_sample": False}
        )

        # Extract structured response
        raw_response = cc.response.strip()
        match = re.search(r"Answer:\s*([A-D])", raw_response)
        extracted_answer = match.group(1).strip() if match else None

        # Get attributions
        result = cc.get_attributions(as_dataframe=True)

        if isinstance(result, pd.io.formats.style.Styler):
            result = result.data

        if isinstance(result, pd.DataFrame):
            qa_id = f"MedBullets df_op4 Q{idx + 1}"
            result["row_index"] = idx
            result["QA_ID"] = qa_id
            result["Extracted_Answer"] = extracted_answer  # Optional: attach model-picked answer

            # Save to individual CSV
            filename = f"{qa_id}.csv".replace(" ", "_")
            result.to_csv(os.path.join(output_dir, filename), index=False)

            print(f"✅ Saved attribution and answer for {qa_id}")
        else:
            print(f"⚠️ Row {idx} returned unexpected type: {type(result)}")

    except Exception as e:
        print(f"❌ Error on row {idx}: {e}")


/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:415: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.


Attributed: Okay, so I've got this question about the conduction speed through the heart. Let me try to break it down. The question is asking which option correctly lists the heart structures from fastest to slowest conduction speed. The options are A, B, C, D.

First, I remember that the heart's electrical signals travel through different parts at varying speeds. The main components involved are the Purkinje fibers, AV node, ventricles, and atria.

I think the Purkinje fibers are the fastest. They're responsible for quickly distributing the electrical impulse to the ventricles, right? So they must conduct the fastest. Then, the ventricles themselves—I believe they conduct faster than the atria because the ventricular tissue is more specialized for rapid contraction.

Wait, what about the AV node? The AV node is a slow point because it's a bottleneck to control the timing between atria and ventricles. So the AV node conducts slower than the ventricles but faster than the atria. Or is i

  0%|                                                                                                  | 0/64 [00:00<?, ?it/s]/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
100%|█████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:09<00:00,  6.43it/s]
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used

✅ Saved attribution and answer for MedBullets df_op4 Q1
Attributed: Okay, so I'm trying to figure out what's going on with this 9-year-old girl. Let me go through the information step by step.

First, the patient presents to the emergency department with a fever and a change in behavior. She had similar symptoms six weeks ago and was treated for an Escherchia coli infection. Also, she had a urinary tract infection (UTI) 10 weeks ago. That's interesting because UTIs can sometimes lead to more serious infections if not properly treated, especially in children.

Her mother says she felt ill last night, and her condition is getting worse. She had a severe headache, stiff neck, which makes me think of meningitis because those are classic symptoms. This morning, she was minimally responsive, vomited several times, and produced dark cloudy urine. Those symptoms are concerning. Dark urine could indicate dehydration or something more serious like kidney issues, but given the other symptoms, I'm

  0%|                                                                                                  | 0/64 [00:00<?, ?it/s]/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
100%|█████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:12<00:00,  5.12it/s]
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used

...

  0%|                                                                                    | 0/64 [00:00<?, ?it/s]/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
100%|███████████████████████████████████████████████████████████████████████████| 64/64 [00:22<00:00,  2.91it/s]
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:410: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation 

✅ Saved attribution and answer for MedBullets df_op4 Q307
Attributed: Okay, so I'm trying to figure out this medical question. Let me read through the context again to make sure I understand everything.

A 26-year-old woman comes to the emergency department because her heart feels like it's "beating out of her chest." She feels otherwise well. Her vital signs are: temperature 97.9°F, blood pressure 124/84 mmHg, pulse 180/min, respirations 22/min, and oxygen saturation 98% on room air. They did an ECG, which is Figure A, but I can't see that. The physician tried vagal maneuvers but they didn't work, so he gave her an IV medication. The effect wore off in seconds.

The question is asking which of the options is a potential side effect of this medication. The options are A. Tachycardia, B. Photosensitivity, C. Flushing, D. Seizure.

First, I need to figure out what's going on with the patient. She has a very fast heart rate—180 bpm. That's significantly tachycardic. The fact that she felt

  0%|                                                                                    | 0/64 [00:00<?, ?it/s]/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
100%|███████████████████████████████████████████████████████████████████████████| 64/64 [00:11<00:00,  5.50it/s]


✅ Saved attribution and answer for MedBullets df_op4 Q308


/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])


## Calculate the Score Specifically

In [ ]:
import os
import pandas as pd

# Set the input folder
input_folder = "Attribution Scores"

# List all CSV files in the folder
csv_files = [f for f in os.listdir(input_folder) if f.endswith(".csv")]

# Process each CSV file
for csv_file in csv_files:
    file_path = os.path.join(input_folder, csv_file)
    
    # Load the attribution file
    df = pd.read_csv(file_path)

    # Safety check: ensure Score column exists
    if "Score" not in df.columns or df["Score"].isnull().all():
        print(f"⚠️ Skipped {csv_file}: no valid 'Score' column.")
        continue

    # Calculate mean score for the current QA
    mean_score = df["Score"].mean()

    # Assign Relevance category
    def get_relevance(score):
        if score == 0:
            return "Irrelevant"
        elif score > mean_score:
            return "Highly Relevant"
        else:
            return "Low Relevant"

    df["Relevance"] = df["Score"].apply(get_relevance)

    # Save the updated file back (overwrite or save new)
    df.to_csv(file_path, index=False)
    print(f"✅ Updated {csv_file} with Relevance labels.")


### The `ContextCiter` class

We can directly instantiate the `ContextCiter` class with a huggingface-style `pretrained_model_name_or_path`, together with a `context`, and a `query` (passed in as strings).

In [17]:
cc = ContextCiter.from_pretrained(model_name_or_path, context, query)

Sliding Window Attention is enabled but not implemented for `sdpa`; unexpected results may be encountered.


Alternatively, we can pass in a `model` and a `tokenizer`, which are instantiated from the `huggingface` library:

In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path)
model = AutoModelForCausalLM.from_pretrained(model_name_or_path)
model.to("cuda")
cc = ContextCiter(model, tokenizer, context, query)

The `response` property of the ContextCiter class contains the response generated by the model. It is lazily generated when you access it.

In [18]:
cc.response

/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.6` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is sa

"Okay, so I need to figure out what type of GPUs the authors used in their paper. Let me start by recalling the context provided. The paper is about the Transformer model, which is a type of attention-based architecture for sequence modeling and machine translation. The user provided a detailed abstract, so I can look for specific details about the hardware used there.\n\nLooking at the abstract, it mentions that the authors achieved their results using eight GPUs, each with P100 architecture. The training took only twelve hours. So, the GPUs were likely high-end computing resources. I should check if there's any mention of specific models or hardware configurations in the abstract. It doesn't explicitly state the model, but it does mention the Transformers and the training setup.\n\nI know that P100 refers to NVIDIA's Pascal architecture, which is a high-end GPU with many cores and high memory bandwidth. Eight of these would be a powerful setup for training models, especially for task

Under the hood, the `ContextCiter` class applies a chat template to the
tokenized context and query, and then uses the model to generate a response.
That response is then stored in the `response` property.

### Attributing the response to sources within the context

To attribute the entire response and present the attributions in a human-readable format, we can use the `get_attributions` method, and pass in `as_dataframe=True`, as well as `top_k` to limit the number of sources to include in the attributions.

In [6]:
results = cc.get_attributions(as_dataframe=True, top_k=5)
results

Attributed: The authors used eight P100 GPUs in their Transformer architecture for training on the WMT 2014 English-to-German translation task.</s>


  0%|                                                                                                                                 | 0/64 [00:00<?, ?it/s]/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:119: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with ch.no_grad(), ch.cuda.amp.autocast():
100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 64/64 [00:03<00:00, 20.68it/s]
/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])


,Score,Source
0,13.856,The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.
1,12.577,"Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU."
2,5.301,"On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature."
3,1.910,"We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely."
4,1.616,"In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output."


`results` is a pandas styler object; to access the underlying dataframe:

In [7]:
results.data

,Score,Source
0,13.856492,The Transformer allows for significantly more ...
1,12.577197,Our model achieves 28.4 BLEU on the WMT 2014 E...
2,5.300751,On the WMT 2014 English-to-French translation ...
3,1.910178,"We propose a new simple network architecture, ..."
4,1.615602,"In this work we propose the Transformer, a mod..."


Alternatively, `.get_attributions()` can return the attribution scores as a `numpy` array, where the `i`th entry corresponds to the attribution score for the `i`th source in the context.

In [8]:
raw_results = cc.get_attributions()
raw_results

Attributed: The authors used eight P100 GPUs in their Transformer architecture for training on the WMT 2014 English-to-German translation task.</s>


array([-0.        , -0.04189325,  1.0423193 , -0.        ,  1.9101778 ,
       -0.527458  , 12.5771966 ,  5.3007509 ,  0.        ,  0.        ,
       -0.        , -0.51422529, -0.26642893,  0.43490208, -0.3266307 ,
        0.        ,  1.05068886, -0.        ,  1.61560223, 13.85649181])

We can then match these attributions to the sources using the `sources` property:

In [9]:
list(zip(cc.sources, raw_results))[:5]

[('Attention Is All You Need', np.float64(-0.0)),
 ('Abstract', np.float64(-0.04189324982279096)),
 ('The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder.',
  np.float64(1.0423192977905273)),
 ('The best performing models also connect the encoder and decoder through an attention mechanism.',
  np.float64(-0.0)),
 ('We propose a new simple network architecture, the Transformer, based solely on attention mechanisms, dispensing with recurrence and convolutions entirely.',
  np.float64(1.9101777961484763))]

### Attributing parts of the response

`.get_attributions()` optionally takes in `start_idx` and `end_idx` to
attribute only a part of the response.

To make it easier to attribute parts of the response, the `ContextCiter` class
has a utility property `response_with_indices` that contains the response annotated with
the index of each word within the response. You can access this with
`cc.response_with_indices`.

In [10]:
print(cc.response_with_indices)

[0]The [4]authors [12]used [17]eight [23]P100 [28]GPUs [33]in [36]their [42]Transformer [54]architecture [67]for [71]training [80]on [83]the [87]WMT [91]2014 [96]English[103]-[104]to[106]-[107]German [114]translation [126]task.</s[134]>


For example, we can attribute a part of the response like so:

In [12]:
start, end = 17, 32
cc.get_attributions(start_idx=start, end_idx=end, as_dataframe=True, top_k=5)

Attributed: eight P100 GPUs


/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])


,Score,Source
0,13.384,The Transformer allows for significantly more parallelization and can reach a new state of the art in translation quality after being trained for as little as twelve hours on eight P100 GPUs.
1,2.426,"On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature."
2,0.316,"Aligning the positions to steps in computation time, they generate a sequence of hidden states ht, as a function of the previous hidden state ht-1 and the input for position t. This inherently sequential nature precludes parallelization within training examples, which becomes critical at longer sequence lengths, as memory constraints limit batching across examples."
3,0.234,"The fundamental constraint of sequential computation, however, remains."
4,0.119,"In this work we propose the Transformer, a model architecture eschewing recurrence and instead relying entirely on an attention mechanism to draw global dependencies between input and output."


In [13]:
start, end = 83, 129
cc.get_attributions(start_idx=start, end_idx=end, as_dataframe=True, top_k=5)

Attributed: the WMT 2014 English-to-German translation task


/data/healthy-ml/scratch/yuexing/hf_env/lib/python3.9/site-packages/context_cite/utils.py:192: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  return df.style.applymap(lambda val: _color_scale(val, max_val), subset=["Score"])


,Score,Source
0,12.398,"Our model achieves 28.4 BLEU on the WMT 2014 English-to-German translation task, improving over the existing best results, including ensembles, by over 2 BLEU."
1,0.646,"On the WMT 2014 English-to-French translation task, our model establishes a new single-model state-of-the-art BLEU score of 41.8 after training for 3.5 days on eight GPUs, a small fraction of the training costs of the best models from the literature."
2,0.224,1 Introduction
3,0.042,The dominant sequence transduction models are based on complex recurrent or convolutional neural networks that include an encoder and a decoder.
4,0.008,"Attention mechanisms have become an integral part of compelling sequence modeling and transduction models in various tasks, allowing modeling of dependencies without regard to their distance in the input or output sequences [2, 19]."
